# QDC ETF Data Refresh and Performance

Review the official 50-instrument universe, check data coverage, and compare recent returns, risk, and co-movement. The backtest and trading workflow use the same CSV format.

Follow the [setup instructions](README.md) to place a local market history CSV in this folder. **Run All** uses the local history by default. Section 1 also has a `REFRESH_FROM_YAHOO` switch for a refresh when you have permission to use the provider's automated feed.

## 1. Load the instrument list and price history

The universe file maps the official QDC instruments to an ISIN and a EUR trading symbol. The symbol determines which listing we query. The loader checks for 50 unique tickers and ISINs and rejects duplicate ticker and date rows.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
from IPython.display import display

from update_qdc_prices import (
    ISSUE_FIELDS,
    PRICE_FIELDS,
    latest_completed_date,
    load_history,
    refresh_prices,
    refresh_summary,
    save_refresh,
)

DATA_DIR = Path.cwd()
REFRESH_FROM_YAHOO = False
THROUGH = latest_completed_date()
SIGNAL_SESSIONS = 5
COMPARISON_SESSIONS = 20
RISK_SESSIONS = 60

assets = pd.read_csv(DATA_DIR / "qdc-50-universe.csv").set_index("ticker")
history = load_history(DATA_DIR)
print(f"{len(history)} EUR listings loaded")
display(assets[["isin", "contest_label", "venue", "currency"]].head())

## 2. Fetch daily data from Yahoo Finance

The shared helper queries Yahoo Finance's chart endpoint once per ticker, with a fallback host if the first request fails. We request daily closing prices, cash dividends, and split events. The refresh overlaps the saved history by one week so a late quote can fill a gap.

The cutoff uses Berlin time. The default permits today's data from 19:00 onward and otherwise stops at yesterday. Review the data quality report before relying on the latest session.

**Optional AI support.** Use the quality report in section 3 to draft a list of ticker and date combinations for review. Check the flagged quotes and corporate actions against the source before using them.

In [ ]:
if REFRESH_FROM_YAHOO:
    updated, issues = refresh_prices(history, THROUGH)
    print(f"Requested Yahoo Finance quotes through {THROUGH}")
else:
    updated = history
    print("Using the saved Yahoo Finance snapshot")

## 3. Check coverage and prepare daily returns

The refresh checks each response's ticker and currency, converts timestamps to the exchange's local dates, and reconciles dividends and splits with saved quotes. Missing closes, source revisions, and items that require review appear in the issue report. When a refresh runs, the merged prices and issues are written to the existing CSV formats.

For the charts, we use sessions with closes for all 50 instruments. For each day, the gross return factor is the current close plus dividend per share, multiplied by the split ratio, then divided by the previous close. This accounts for cash distributions and changes in share count. The common calendar gives each instrument the same comparison dates.

When an observed date has incomplete coverage, the return between the two retained closes spans the gap. Check the omitted dates listed below when interpreting the daily volatility and correlation estimates.

In [ ]:
issues_path = DATA_DIR / "qdc-50-data-issues.csv"
if REFRESH_FROM_YAHOO:
    price_path, issues_path = save_refresh(DATA_DIR, updated, issues, THROUGH)
    new_rows, failed_tickers, _ = refresh_summary(history, updated, issues)
    print(f"{new_rows} new rows | {len(failed_tickers)} tickers with fetch or corporate action issues")
    print(f"Saved {price_path.name} and {issues_path.name}")

if issues_path.exists():
    quality = pd.read_csv(issues_path, usecols=ISSUE_FIELDS)
    print(f"Rows in the saved refresh report: {len(quality)}")
    if not quality.empty:
        display(quality.head(10))
        if len(quality) > 10:
            print(f"First 10 shown. See {issues_path.name} for the full report.")

quotes = pd.DataFrame(
    (row for rows in updated.values() for row in rows.values()), columns=PRICE_FIELDS
)
quotes["date"] = pd.to_datetime(quotes["date"])
for column in ("close_eur", "dividend_eur", "split_multiplier"):
    quotes[column] = pd.to_numeric(quotes[column], errors="raise")

all_closes = quotes.pivot(index="date", columns="ticker", values="close_eur")
closes = all_closes.reindex(columns=assets.index).dropna()
dividends = quotes.pivot(index="date", columns="ticker", values="dividend_eur").reindex_like(closes)
splits = quotes.pivot(index="date", columns="ticker", values="split_multiplier").reindex_like(closes)
if dividends.isna().any().any() or splits.isna().any().any():
    raise ValueError("A complete close is missing dividend or split information")

returns = ((closes + dividends) * splits / closes.shift() - 1).iloc[1:]
required_sessions = max(SIGNAL_SESSIONS, COMPARISON_SESSIONS, RISK_SESSIONS)
if len(returns) < required_sessions:
    raise ValueError(f"At least {required_sessions + 1} common closes are required")
omitted = all_closes.loc[closes.index[-required_sessions - 1]:].index.difference(closes.index)
print(f"{len(quotes):,} saved quotes | {len(closes):,} common sessions")
print(f"Latest complete close: {closes.index[-1].date()}")
if len(omitted):
    print("Observed dates omitted for incomplete coverage:", ", ".join(omitted.strftime("%Y-%m-%d")))

## 4. Compare returns and risk across all 50 instruments

The short window matches the existing five-session signal. The 20-session return gives a second view of recent performance. The 60-session risk window estimates annualized volatility from daily return variation and maximum drawdown from the deepest decline below a previous peak. These windows are starting values for team discussion.

The ranking chart shows the strongest and weakest instruments on the short window. The scatter shows every instrument. Look for names whose 20-session performance came with unusually high volatility. Then check the table to see whether the short-window ranking tells the same story.

In [ ]:
def cumulative_return(daily_returns):
    return (1 + daily_returns).prod() - 1


signal_label = f"{SIGNAL_SESSIONS}-session return"
comparison_label = f"{COMPARISON_SESSIONS}-session return"
risk_returns = returns.tail(RISK_SESSIONS)
risk_path = (1 + risk_returns).cumprod()
risk_peak = risk_path.cummax().clip(lower=1)
metrics = pd.DataFrame({
    "Instrument": assets["contest_label"],
    signal_label: cumulative_return(returns.tail(SIGNAL_SESSIONS)),
    comparison_label: cumulative_return(returns.tail(COMPARISON_SESSIONS)),
    "Annualized volatility": risk_returns.std() * 252**0.5,
    "Maximum drawdown": (risk_path / risk_peak - 1).min(),
}).sort_values(signal_label, ascending=False)

date_label = closes.index[-1].strftime("%d %b %Y")
edges = pd.concat([metrics.head(5), metrics.tail(5)]).sort_values(signal_label)
fig, ax = plt.subplots(figsize=(8.4, 4.4), layout="constrained")
ax.barh(edges.index, edges[signal_label],
        color=["#bc4d39" if value < 0 else "#147c76" for value in edges[signal_label]])
ax.axvline(0, color="#444444", linewidth=0.8)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set(title=f"Strongest and weakest {SIGNAL_SESSIONS}-session returns",
       xlabel=f"Cumulative total return through {date_label}")
plt.show()

fig, ax = plt.subplots(figsize=(8.4, 4.9), layout="constrained")
ax.scatter(metrics["Annualized volatility"], metrics[comparison_label],
           color="#147c76", alpha=0.65, s=31, label="50 instruments")
labels = {"EUNL.DE": "WORLD", "G2X.DE": "GOLD MINERS", "LYOR.DE": "OVERNIGHT CASH"}
for ticker, label in labels.items():
    volatility, total_return = metrics.loc[ticker, ["Annualized volatility", comparison_label]]
    ax.scatter(volatility, total_return, marker="D", color="#203353", s=39, zorder=3)
    ax.annotate(label, (volatility, total_return), xytext=(6, 6), textcoords="offset points", fontsize=9)
ax.axhline(0, color="#888888", linewidth=0.8)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(title=f"Recent return versus volatility through {date_label}",
       xlabel=f"Annualized volatility from {RISK_SESSIONS} sessions",
       ylabel=f"{COMPARISON_SESSIONS}-session cumulative total return")
plt.show()

In [ ]:
metric_columns = list(metrics.columns[1:])
display(metrics.style.format({column: "{:+.2%}" for column in metric_columns})
        .set_table_attributes('style="font-size: 0.9em"')
        .set_caption(f"All 50 instruments ranked by {signal_label.lower()} through {date_label}"))

## 5. Compare paths and check overlap

The line chart sets each instrument to 100 at the start, so the paths show comparable gains and losses. WORLD provides a broad equity reference. The heatmap uses 60 shared daily returns and a small group of instruments to make co-movement easier to see. A high positive correlation means the instruments tended to rise and fall together in this sample. Correlation can change over time.

In [ ]:
comparison_names = {
    "EUNL.DE": "WORLD",
    "SXR8.DE": "S&P 500",
    "SXRV.DE": "NASDAQ",
    "SEC0.DE": "SEMIS",
    "PPFB.DE": "GOLD",
    "G2X.DE": "GOLD MINERS",
    "LYOR.DE": "OVERNIGHT CASH",
}
paths_to_plot = ["EUNL.DE", "SXRV.DE", "G2X.DE", "LYOR.DE"]
period_returns = returns[paths_to_plot].tail(COMPARISON_SESSIONS)
base_date = closes.index[-COMPARISON_SESSIONS - 1]
indexed_paths = pd.concat([
    pd.DataFrame(100.0, index=[base_date], columns=paths_to_plot),
    (1 + period_returns).cumprod() * 100,
]).rename(columns=comparison_names)

fig, ax = plt.subplots(figsize=(8.4, 4.4), layout="constrained")
for label, color in zip(indexed_paths, ["#203353", "#147c76", "#bc4d39", "#717985"]):
    ax.plot(indexed_paths.index, indexed_paths[label], label=label, color=color, linewidth=2)
ax.axhline(100, color="#888888", linewidth=0.8, linestyle="--")
ax.set(title=f"Selected instruments over {COMPARISON_SESSIONS} common sessions",
       ylabel="Total return index (start = 100)")
ax.legend(loc="best", frameon=False, ncols=2)
ax.tick_params(axis="x", labelrotation=25)
plt.show()

corr = risk_returns[list(comparison_names)].rename(columns=comparison_names).corr()
fig, ax = plt.subplots(figsize=(7.7, 5.7), layout="constrained")
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr)), corr.index)
for row in range(len(corr)):
    for column in range(len(corr)):
        value = corr.iat[row, column]
        ax.text(column, row, f"{value:.2f}", ha="center", va="center",
                color="white" if abs(value) > 0.65 else "#151515", fontsize=9)
ax.set_title(f"Daily return correlations over {RISK_SESSIONS} common sessions")
fig.colorbar(image, ax=ax, label="Pearson correlation", shrink=0.8)
plt.show()

## 6. Discuss the rules as a team

The trading workflow demo uses five-session momentum to pick the core equity ETF and recent volatility to pick the companion. It reserves 1% of the portfolio as cash and splits the investable amount 70/30.

| Decision | What to discuss |
|---|---|
| Signal | Should recent strength be measured over 5 sessions, 20 sessions, or both? |
| Selection | Should the companion be picked by its own volatility or by its effect on the whole portfolio? |
| Position size | How much can one instrument or theme account for, and when should we keep extra cash? |
| Trading | How large should a change in target weight be before we place an order? |

To compare rankings on the same snapshot, set `REFRESH_FROM_YAHOO = False`, then run with `SIGNAL_SESSIONS` set to 5 and again with it set to 20. Keep `COMPARISON_SESSIONS` at 20. Use the [strategy backtest](qdc-etf-50-backtest.ipynb) to test how a change affects portfolio returns and drawdown across historical windows.